In [1]:
import os

from src.utils.logging_config import get_project_root
from src.utils.settings import Settings

default_path = get_project_root().as_posix()
os.chdir(default_path)
settings = Settings(_yaml_file="./config/runs/env_params.yaml")

settings.model_seizure.optuna_parameters.study_name
run_tensorboard_optuna_logs = f"reports/tensorboard/{settings.name}/logs_optuna"
run_tensorboard_logs = ""  # f"reports/tensorboard/{settings.name}/logs_optuna/{run_history['model_seizure']['optuna_parameters']['study_name']}"
run_code_carbon_csv = ""
study_name = f"{settings.model_seizure.optuna_parameters.study_name}"
run_study_path = f"reports/tensorboard/{settings.name}/logs_optuna/{settings.model_seizure.optuna_parameters.study_name}"
run_study_db_dir = f"{settings.name}/{study_name}"
run_study_db_path = f"{run_study_db_dir}/optuna.db"

# Tensorboard

In [ ]:
!tensorboard --logdir='{default_path}{run_tensorboard_logs}' --host localhost --port 8888

In [ ]:
%load_ext tensorboard

# Code carbon

In [ ]:
!carbonboard --filepath="{default_path}{run_code_carbon_csv}" --port=3333

# Plots

# Optuna

In [ ]:
# Only to be used with pickled optuna studies, not with optuna dbs

import joblib
import optuna

# to load optuna
study_test = joblib.load(default_path + '/' + run_study_path)

study2 = optuna.create_study(study_name=study_test["study_name"], direction=study_test["direction"])

# Add the loaded trials to the study
study2.add_trials(study_test["trials"])
# study2 = joblib.load(f"../reports/tensorboard/{experiment_name}/logs_optuna/{study_name}_study")
print("Best trial until now:")
print(" Value: ", study2.best_trial.value)
print(" Params: ")
for key, value in study2.best_trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
# Only to be used with optuna dbs, not with pickled optuna studies

import optuna

db_url = f"sqlite:///{default_path}/reports/runs/{run_study_db_path}"
study = optuna.load_study(study_name=study_name, storage=db_url)
print("Best trial until now:")
print(" Value: ", study.best_trial.value)
print(" Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
#!optuna studies --storage mysql://root@localhost/example

In [ ]:
!optuna-dashboard "sqlite:///{default_path}{run_study_db_path}"

# MLFLOW

In [ ]:
import sqlite3

db_path = os.path.abspath("reports/mlflow.db")
conn = sqlite3.connect(db_path)
c = conn.cursor()

# Select all data from a specific table
c.execute("SELECT run_uuid, artifact_uri FROM runs LIMIT 5")
data = c.fetchall()
print(data)

conn.close()

!mlflow server --host 127.0.0.1 --port 8089

# Best model

## Load data

## Load Model

## Confusion Matrix

# Load docker container with all services - commands also in makefile

In [ ]:
!TENSORBOARD_LOG_DIR='{default_path}{run_tensorboard_logs}' \
    OPTUNA_LOG_DIR='{default_path}/reports/runs/{run_study_db_dir}' \
    docker compose -f ./CI-CD/mlops_compose_stack.yaml up -d

[+] up 0/3
 ⠋ Container ci-cd-mlflow-1      Recreate                                   0.1s
 ⠋ Container ci-cd-optuna-1      Recreate                                   0.1s
 ⠋ Container ci-cd-tensorboard-1 Recreate                                   0.1s
[+] up 0/3
 ⠙ Container ci-cd-mlflow-1      Starting                                   0.2s
 ⠙ Container ci-cd-optuna-1      Starting                                   0.2s
 ⠙ Container ci-cd-tensorboard-1 Starting                                   0.2s
[+] up 0/3
 ⠹ Container ci-cd-mlflow-1      Starting                                   0.3s
 ⠹ Container ci-cd-optuna-1      Starting                                   0.3s
 ⠹ Container ci-cd-tensorboard-1 Starting                                   0.3s
[+] up 2/3
 ✔ Container ci-cd-mlflow-1      Started                                    0.3s
 ✔ Container ci-cd-optuna-1      Started                                    0.3s
 ⠸ Container ci-cd-tensorboard-1 Starting                        

In [ ]:
!TENSORBOARD_LOG_DIR='{default_path}{run_tensorboard_logs}' \
    OPTUNA_LOG_DIR='{default_path}/reports/runs/{run_study_db_dir}' \
    docker compose -f ./CI-CD/mlops_compose_stack.yaml down